# Weekly Update Generator - AgentCore 배포

## 개요

이 튜토리얼에서는 Amazon Bedrock AgentCore Runtime을 사용하여 자동화된 주간 상태 보고서 생성기를 구축하고 배포하는 방법을 알아봅니다. 에이전트는 여러 소스(team update, meeting note, metric, bug tracker)에서 데이터를 수집하고 분석과 시각화를 수행한 다음 종합 보고서를 S3에 업로드합니다.

### 아키텍처 및 파일

![아키텍처 다이어그램](./images/architecture.png)

- **Agent 구현**: `01_weekly_report_generator_async/agent/agent.py` - Bedrock Agent Core 애플리케이션, agent 구성, entrypoint를 정의하는 주요 agent 파일
- **Tools**: `01_weekly_report_generator_async/agent/tools.py` - 데이터 읽기, 분석, 시각화, S3 업로드를 위한 16개 tool 포함
- **Demo Data**: 다음 항목이 포함된 `01_weekly_report_generator_async/demo_data/` 디렉터리
  - `project_status/` - 프로젝트 진행 상황과 blocker가 포함된 CSV 파일
  - `team_updates/` - 개별 팀원의 update가 포함된 Markdown 파일
  - `metrics/` - 현재 및 과거 KPI 데이터가 포함된 CSV 파일
  - `issues/` - bug tracker 데이터가 포함된 JSON 파일
  - `meeting_notes/` - meeting summary가 포함된 Markdown 파일

### 작동 방식

1. **데이터 수집** - 에이전트가 최신 주차 데이터를 동적으로 검색하고 읽음
   - 디렉터리에서 `projects_week_XX.csv`, `kpis_week_XX.csv`, `bug_tracker_week_XX.json` 같은 pattern과 일치하는 파일 검색
   - 검색된 가장 최근 주차 번호를 자동으로 사용
   - 모든 팀원 Markdown 파일과 meeting note 읽기

2. **데이터 분석** - 에이전트가 지능형 분석 수행
   - 모든 소스의 데이터 품질과 완전성 검증
   - 정보 교차 확인(예: 상태 파일과 team update의 프로젝트 이름 비교)
   - 사기 저하 문제를 감지하도록 team update에 sentiment analysis 수행
   - 프로젝트 상태, blocker, bug severity를 기준으로 risk score 계산

3. **시각화 생성** - matplotlib을 사용하여 PNG chart 생성
   - bug severity 분포(pie/bar chart)
   - 과거 비교를 포함한 KPI metric 추세
   - 프로젝트 timeline 및 진행 상황 시각화
   - 팀 velocity chart
   - 주요 metric의 predictive forecast 모델

4. **보고서 종합** - 다음 항목으로 구성된 structured Markdown 보고서 작성
   - 주요 highlight와 우려 사항이 포함된 executive summary
   - 프로젝트, team update, KPI, bug, meeting에 관한 상세 섹션
   - risk analysis 및 blocker
   - action item 및 다음 주 우선순위

5. **S3 업로드** - S3에 자동 업로드
   - 최종 Markdown 보고서
   - 생성된 모든 chart image
   - 쉽게 가져올 수 있도록 S3 bucket에서 날짜별로 구성

### 튜토리얼 세부 정보

| 항목                | 세부 정보                                                                        |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | Asynchronous agent                                                        |
| Agent 유형          | Single                                                                           |
| Agentic Framework   | Strands Agents                                                                   |
| LLM 모델            | Anthropic Claude Sonnet 4.5                                                       |
| 튜토리얼 구성 요소  | multi-tool agent, 데이터 분석, 시각화, S3 통합, AgentCore Runtime                 |
| 튜토리얼 분야       | 비즈니스 운영 및 보고                                                            |
| 예제 난이도         | 중급                                                                              |
| 사용 SDK            | Amazon BedrockAgentCore Python SDK, boto3, matplotlib, scikit-learn              |

### 튜토리얼 아키텍처

이 튜토리얼에서는 asynchronous reporting agent를 AgentCore Runtime에 배포하는 방법을 보여줍니다. 에이전트는 16개의 서로 다른 tool을 조율하여 종합 주간 상태 보고서를 자동으로 생성합니다.

### 튜토리얼 주요 기능

* asynchronous multi-tool agent를 Amazon Bedrock AgentCore Runtime에 호스팅
* Amazon Bedrock 모델(Claude Sonnet 4) 사용
* Strands Agents SDK 사용
* 데이터 저장 및 보고서 전달을 위한 S3 통합

### 배포

에이전트는 다음과 같이 사용할 수 있는 Bedrock Agent Core 애플리케이션으로 실행됩니다.
- 테스트를 위해 로컬에서 호출
- production 용도로 AWS Lambda에 배포
- async task tracking을 사용하는 API로 호출(background에서 처리하는 동안 즉시 반환)
- HEALTHY 또는 HEALTHY_BUSY 상태를 보고하는 ping endpoint를 통해 monitoring


## 사전 요구 사항
- Amazon Bedrock AgentCore 액세스 권한이 있는 AWS 계정
- AWS credentials
- Python 3.12+
- demo data와 보고서를 저장할 S3 bucket

## 설정 및 Import

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
import time
import json
import boto3
from datetime import datetime, timedelta

print("✅ Imports successful!")

## 배포 전 구성

In [ ]:
boto_session = Session()
region = boto_session.region_name
agent_name = "weekly_update_generator"

# TODO: S3 bucket 이름으로 교체
S3_BUCKET = "YOUR-BUCKET-NAME"
S3_PREFIX = "demo_data"

print(f"📍 Region: {region}")
print(f"🤖 Agent name: {agent_name}")
print(f"🪣 S3 Bucket: {S3_BUCKET}")

## 1단계: Demo Data 업데이트 및 S3 업로드

In [ ]:
# update script 실행
!python update_demo_dates.py --bucket {S3_BUCKET} --prefix {S3_PREFIX}

## 2단계: AgentCore에 Agent 배포

CreateAgentRuntime operation은 container image, environment variable, encryption 설정을 지정할 수 있는 포괄적인 구성 옵션을 지원합니다. protocol 설정(HTTP, MCP)과 authorization mechanism도 구성하여 client가 에이전트와 통신하는 방식을 제어할 수 있습니다.

이 튜토리얼에서는 Amazon Bedrock AgentCore Python SDK를 사용하여 artifact를 손쉽게 package하고 AgentCore Runtime에 배포합니다.

### Async Agent 구현

`01_weekly_report_generator_async/agent/agent.py` 파일은 장시간 실행되는 보고서 생성 task를 처리하는 asynchronous agent를 구현합니다. 에이전트는 호출되면 task ID를 즉시 반환하고 background thread에서 request를 처리합니다. 따라서 호출자는 전체 보고서 생성이 완료될 때까지 기다리지 않고 계속 진행할 수 있습니다.

에이전트는 counter를 사용하여 활성 task를 추적하고 에이전트 상태를 보고하는 `@app.ping` handler를 구현합니다.

### /ping Endpoint 이해

`/ping` endpoint는 async agent monitoring에 중요하며 다음 두 상태 중 하나를 반환합니다.

- **`{"status": "Healthy"}`** - 에이전트가 idle 상태이며 새 task를 받을 준비가 됨
- **`{"status": "HealthyBusy"}`** - 에이전트가 task를 처리 중이지만 계속 응답 가능

이를 통해 client는 다음 작업을 수행할 수 있습니다.
- 에이전트를 polling하여 task 완료 시점 확인
- 진행 중인 작업을 방해하지 않고 agent health monitoring
- agent availability에 따라 retry logic 구현

자세한 내용은 [Asynchronous Agents 문서](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-long-run.html)를 참조하세요.

### AgentCore Runtime 배포 구성
먼저 starter toolkit을 사용하여 entrypoint, 방금 생성한 execution role, requirements 파일로 AgentCore Runtime 배포를 구성합니다. 또한 시작할 때 Amazon ECR repository를 자동으로 생성하도록 starter toolkit을 구성합니다.

configure 단계에서 애플리케이션 코드를 기반으로 Dockerfile이 생성됩니다. 에이전트가 Runtime에서 tool에 액세스할 수 있도록 tool도 함께 package됩니다.

![](images/configure.png)

In [ ]:
# 구성을 위해 agent 디렉터리로 이동
import os

os.chdir("agent")
print(f"Working directory: {os.getcwd()}")

### Agent 구성

In [ ]:
agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
)

print("✅ Agent configured")
response

### AgentCore Runtime에 Agent 시작

Dockerfile이 준비되었으므로 에이전트를 AgentCore Runtime에 시작합니다. 이 과정에서 Amazon ECR repository와 AgentCore Runtime이 생성됩니다.

![](images/launch.png)

In [ ]:
launch_result = agentcore_runtime.launch()

print("🚀 Agent launched")
print(f"Agent ARN: {launch_result.agent_arn}")
launch_result

# Notebook 디렉터리로 돌아가기
os.chdir("..")
print(f"Changed back to: {os.getcwd()}")

## 3단계: Agent가 준비될 때까지 대기

에이전트를 성공적으로 호출할 수 있도록 배포 상태를 확인합니다.

In [ ]:
status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]
end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]

print(f"Status: {status}")

while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]
    print(f"Status: {status}")

print(f"\n✅ Agent is {status}")

## 4단계: Execution Role에 S3 권한 추가

이 튜토리얼에서는 Amazon S3 bucket에 저장된 데이터를 읽고 씁니다. 에이전트를 시작했으므로 S3 bucket 액세스 권한을 에이전트에 부여해야 합니다.

In [ ]:
# agent runtime에서 execution role 가져오기
agent_runtime_id = launch_result.agent_arn.split("/")[-1]
print(f"Agent Runtime ID: {agent_runtime_id}")

agentcore_client = boto3.client("bedrock-agentcore-control", region_name=region)


response = agentcore_client.get_agent_runtime(agentRuntimeId=agent_runtime_id, agentRuntimeVersion="1")

execution_role_arn = response.get("roleArn")

execution_role_name = execution_role_arn.split("/")[-1]
print(f"✓ Execution role: {execution_role_name}")

iam_client = boto3.client("iam")
policy_document = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["s3:GetObject", "s3:PutObject", "s3:ListBucket"],
            "Resource": [f"arn:aws:s3:::{S3_BUCKET}", f"arn:aws:s3:::{S3_BUCKET}/*"],
        }
    ],
}

iam_client.put_role_policy(
    RoleName=execution_role_name,
    PolicyName="WeeklyReportsS3Access",
    PolicyDocument=json.dumps(policy_document),
)

print("\n✅ S3 permissions added!")
print(f"   Role: {execution_role_name}")
print(f"   Bucket: {S3_BUCKET}")

## 5단계: Agent 호출

마지막으로 에이전트를 호출하여 주간 보고서를 생성합니다.

In [ ]:
import time
import json

# 현재 주차 정보 가져오기
today = datetime.now()
monday = today - timedelta(days=today.weekday())

print(f"🚀 Starting async agent invocation for week of {monday.strftime('%B %d, %Y')}...\n")

# 에이전트 호출
start_time = time.time()
invoke_response = agentcore_runtime.invoke(
    {"prompt": f"Generate the weekly status report for the week of {monday.strftime('%B %d, %Y')}."}
)

print("✅ Agent invocation started")

# response에서 task 정보 추출
if "response" in invoke_response and invoke_response["response"]:
    task_info = json.loads(invoke_response["response"][0])
    task_id = task_info.get("task_id")
    print(f"📋 Task ID: {task_id}")
    print(f"📊 Initial Status: {task_info.get('status')}")
    print(f"💬 {task_info.get('message')}\n")

print("⏳ Polling agent status...\n")

poll_count = 0
time.sleep(5)  # 에이전트 시작 시간 제공

while True:
    poll_count += 1
    elapsed = time.time() - start_time

    try:
        # 에이전트를 ping하여 busy 상태 확인"
        ping_response = agentcore_runtime.invoke({"method": "ping", "payload": {}})

        if "response" in ping_response and ping_response["response"]:
            response_data = json.loads(ping_response["response"][0])
            health_status = response_data.get("status", "Unknown")
            active_tasks = response_data.get("active_tasks", 0)

            print(f"Poll #{poll_count} ({elapsed:.1f}s): {health_status} (active tasks: {active_tasks})")

            # 에이전트가 Healthy 상태인지 확인(활성 task 없음)
            if health_status == "Healthy":
                print(f"\n✅ Task completed in {elapsed:.1f} seconds")
                break

    except Exception as e:
        print(f"Poll #{poll_count} ({elapsed:.1f}s): Error - {e}")

    if elapsed > 600:
        print(f"\n⚠️ Timeout after {elapsed:.1f} seconds")
        break

    time.sleep(5)

# 보고서 output path 출력
week_num = today.isocalendar()[1]
year = today.year
week_folder = f"{year}/week_{week_num:02d}_{monday.strftime('%Y-%m-%d')}"
print(f"\n📁 Report should be at: s3://{S3_BUCKET}/weekly_reports/{week_folder}/weekly_report.md")

## 6. Agent Output 보기

에이전트가 주간 보고서를 생성하면 S3의 s3:/{your-bucket-name}/weekly_reports/{year}/{week}/weekly_report.md 경로에서 확인할 수 있습니다.

![](images/report.png)

## 리소스 정리(선택 사항)
이제 생성된 AgentCore Runtime을 정리합니다.

In [ ]:
launch_result.ecr_uri, launch_result.agent_id, launch_result.ecr_uri.split("/")[1]

agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id,
)
print("Deleting Runtime")

In [ ]:
repo_name = launch_result.ecr_uri.split("/")[-1].split(":")[0]

ecr_client = boto3.client("ecr", region_name=region)
response = ecr_client.delete_repository(repositoryName=repo_name, force=True)
print("Deleting ECR")